<a href="https://colab.research.google.com/github/Kamalashrinithi19/kamalashrinithi-codeboosters-2026/blob/main/Day7/Day7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing the Machinery:

What it does: This installs the two core external libraries needed for vector operations. The -q flag stands for "quiet," which hides the messy installation progress bars.

chromadb: An open-source vector database. Think of it like SQLite, but instead of storing text or integers in tables, it stores arrays of numbers (embeddings) and quickly calculates the geometric distance between them.

sentence-transformers: A framework built on top of PyTorch and Hugging Face that lets you load pre-trained models (like BERT or MiniLM) to convert your raw strings into these number arrays.

In [ ]:
# Libraries to istall: chromadb- vector darabase(like SQLite but for AI embeddings)
# sentence-transformers: converts text to 384-dimensional number vector

!pip install chromadb sentence-transformers -q

# -q means 'quiet mode' - hides the nessy installation process
# You will see a progress bar. Wait untit it says installed completely

print('Installation compelte!')

In [ ]:
import pandas as pd # for loading and exploring the dataset
import numpy as np      #for working with numerical arrays

# new library: sentence-transformers
# SentenceTransformer: the class we used to load the embeddding model
from sentence_transformers import SentenceTransformer

# new library: chromadb
# chromadb: the vecotor database package
import chromadb

print('All bibraries imported successfully!')
print(f"ChromaDB version: {chromadb.__version__}")

In [ ]:
# DEMO: Keyword search vs Semantic search

# Imagine we have a samlll collection of documents about data
documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportartion",
    "Car and Trucks ar popular Automobiles",
    "SQL is used to query databases",
    "Machine Learning trains model on data"
]

query_keyword = "vehicle"
print('='*10)
print(f"KEYWORD SEARCH for: {query_keyword}")
print('='*10)

for i, doc in enumerate(documents):
  if query_keyword.lower() in doc.lower():
    print(f"\n  FOUND\t\t[doc_{i}]: {doc}")
  else:
    print(f"\n  MISSED\t[doc_{i}]: {doc}")

print()
print("PROBLEM doc_2 talks about 'Cars and trucks' - which ARE vehicle!")
print("But keywordsearch missed it because it searched  for the exact word")

In [ ]:
# More examples of keyword search failures
# Ther  are all cases where meaning mathces but word or not

failure_example = [
    {"query": "I feel sick", "misses": "I am unwell, patient has fever"},
    {"query": "How to cook rice", "misses": "Steps to prepare rise"},
    {"query": "Vehicle speed", "misses": "car acceleration, automobile velocity"},
    {"query": "ML model accuracy", "misses": "performance of machien learning model"}
]

for ex in failure_example:
  print(f'Query: {ex["query"]}')
  print(f'Answer: {ex["misses"]}')
  print('-'*19)

In [ ]:
# to get o.p - 384-dimensional vectors
# Note: first run downloads the model (~80MB)

print("Loading embedding model... (may take 1-2 mins on first run)")

model= SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded successfully!")
print(f"Model produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")

# 📝 Core Notes: How Text Embeddings Work

## 1. What is an Embedding?

* An embedding is a process that converts raw text into a fixed-size array of numbers (a **vector**).
* For the `all-MiniLM-L6-v2` model, this vector **always** contains exactly **384 numbers**, regardless of whether the input is a single word, a short sentence, or a full paragraph.
* **Crucial Rule:** The number 384 represents **dimensions (conceptual traits)**, not word counts.

---

## 2. The 3-Step Pipeline Under the Hood

When you run `model.encode(sentence)`, the text goes through three major phases:

### A. Tokenization & Padding

* **Tokenization:** Words are broken down into smaller pieces called tokens and assigned numeric IDs.
* **Padding:** Neural networks require fixed-size inputs to perform matrix calculations. The model pads shorter text with `[PAD]` tokens up to its maximum capacity (e.g., 256 or 512 tokens) so the data structure matches perfectly.

### B. Transformer Layers (Contextual Generation)

* The tokens pass through the neural network layers.
* Every token receives its own 384-dimensional vector that describes its meaning *based on the words surrounding it*.
* At this stage, the data is a 2D matrix of shape: `(Total Tokens × 384 Dimensions)`.

### C. Mean Pooling (The Compression Step)

* To get a single vector for the entire sentence, the model applies **Mean Pooling**.
* It ignores all the useless `[PAD]` tokens using an attention mask.
* It calculates the mathematical average of all the remaining meaningful token vectors.
* **Result:** The 2D matrix is compressed into a single flat 1D vector of shape `(384,)`.

---

## 3. What Do the 384 Numbers Actually Represent?

* **Abstract Concepts, Not Words:** Individual numbers do not map to literal strings (e.g., `0.054` does not mean "feline" or "data").
* **Conceptual Sliders:** Each dimension acts as a nameless mathematical tracker for an abstract trait or feature that the model discovered on its own during training (e.g., tracking properties like "is technical", "has a casual tone", "implies motion", etc.).
* **Values indicate alignment:**
* **High Positive Value:** Strong alignment with that concept.
* **Near Zero Value:** The concept is completely irrelevant to the text.
* **Negative Value:** The text represents the conceptual opposite of that trait.



---

## 4. How Semantic Search Works "By Itself"

* **No Manual Dictionaries Needed:** The model inherently understands synonyms because it was pre-trained on billions of sentences.
* **Shared Coordinates:** Because words like *"cat"* and *"feline"* or *"ETL"* and *"data pipelines"* appear in identical contexts during training, the model naturally assigns them highly similar scores across those 384 dimensions.
* **Distance Over Matches:** Semantic search tools (like ChromaDB) don't match strings. They check how close the 384 coordinates are to each other in vector space using distance math (like cosine similarity). If the vectors point in the same direction, they are surfaced as a match.

In [ ]:
# 2. Generate your 1st embedding

# Define a single sentence to embed
sentence = "ETL is used to clean and transform data"

# model.encode() converts text to a vector
embedding = model.encode(sentence)

print(f"Input sentence: {sentence}")
print(f"Embedding type: {type(embedding)}")
print(f"Embedding size: {len(embedding)}")
print(f"Embedding shape: {embedding.shape}")
print(f"First 10 numbers: {embedding[0:10]}")
print(f"Min value: {embedding.min()}")
print(f"Max value: {embedding.max()}")

## 5. Semantic Search: Embed all Documents

In [ ]:
# Embed all documents
document_embeddings = model.encode(documents)

print(f"Number of document embeddings: {len(document_embeddings)}")
print(f"Shape of each embedding: {document_embeddings[0].shape}")

## 6. Semantic Search: Embed the Query and Calculate Similarity

We will use `util.cos_sim` from `sentence_transformers` for cosine similarity, which measures the cosine of the angle between two non-zero vectors. A value closer to 1 indicates higher similarity.

In [ ]:
from sentence_transformers import util

# Define a semantic query
query_semantic = "cars and vehicles"

# Embed the query
query_embedding = model.encode(query_semantic)

# Calculate cosine similarity between the query and all document embeddings
cosine_scores = util.cos_sim(query_embedding, document_embeddings)[0]

# Combine documents and their scores
semantic_results = []
for i, score in enumerate(cosine_scores):
    semantic_results.append({"document": documents[i], "score": score.item()})

# Sort results by score in descending order
semantic_results = sorted(semantic_results, key=lambda x: x["score"], reverse=True)

print(f"Semantic Search for: '{query_semantic}'")
print('='*30)
threshold=0.5
for res in semantic_results:
    if res['score'] < threshold:
        flag='MISSED'
    else:
        flag='FOUND'
    print(f"Score: {res['score']:.4f}\t {flag}    \tDocument: {res['document']}")

Notice how `Car and Trucks are popular Automobiles` (doc 2) is now ranked higher, even though it doesn't contain the exact word "vehicle". This is the power of semantic search!

In [ ]:
from sentence_transformers import util
sentences = [
    # "Machine learning trains models on labeled data",
    # "AI learn patterns from exeamples",
    # "I enjoy eating watermelon",
    "I love cats",
    "😺",
    "😖😭🐾"
]

embeddings=model.encode(sentences)

def cosine_similarity(s1, s2):
  return util.cos_sim(s1, s2).item()

sim_01 = cosine_similarity(embeddings[0], embeddings[1])
sim_02 = cosine_similarity(embeddings[0], embeddings[2])

print("YOUR EXPERIMENT RESULLTS")
print('-'*30)
print(f'Sentence A: {sentences[0]}')
print(f'Sentence B: {sentences[1]}')
print(f'Sentence C: {sentences[2]}')
print(f'Similarity (A vs B): {sim_01:.4f}')
print(f'Similarity (A vs C): {sim_02:.4f}')

In [ ]:
sentence =[
    "Machine Learning is used to predict the ouput based on trained data",
    "Types : Supervised and Unsupervised Learning",
    "I enjoy learning in the classroom",
]

embedding = model.encode(sentence)
query_keyword = "Machine Learning"
print(f"Query: {query_keyword}\n")

for i,doc in enumerate(sentence):
  print(f"Doc{i}: {doc}")
  embedding_query = model.encode(query_keyword.lower())
  embedding_doc = model.encode(doc.lower())
  similarity = util.cos_sim(embedding_query, embedding_doc).item()
  if similarity > 0.4:
    print(f" FOUND [doc_{i}]: {doc} (similarity={similarity:.2f})\n")
  else:
    print(f" MISSED [doc_{i}]: {doc} (similarity={similarity:.2f})\n")

In [ ]:
chroma_client =chromadb.Client()
collection=chroma_client.get_or_create_collection("demo_notes")
print("ChromaDB client created (in-memory mode)")


print(f"Collection name:demo_notes ")
print(f"Documents in collection :{collection.count()}")

In [ ]:
sample_docs = [
    "ETL is data transformation pipeline",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automoblies",
    "Extracts, Transform and Load",
    "Machine learning trains models o data"
]

sample_ids = ['doc001', 'doc002', 'doc003', 'doc004', 'doc005']

sample_metadata = [
    {"subject": "Data Engineering", "topic": "ETL"},
    {"subject": "Transportation", "topic": "Vehicles"},
    {"subject": "Transportation", "topic": "Automobiles"},
    {"subject": "Data Engineering", "topic": "Data Transformation"},
    {"subject": "Machine Learning", "topic": "Models"}
]

sample_embeddings = model.encode(sample_docs).tolist()

collection.add(
    documents=sample_docs,
    embeddings=sample_embeddings, # Pass the generated embeddings
    ids=sample_ids,
    metadatas=sample_metadata
)

print("Documents added successfully")
print(f"Number of documents in collection: {collection.count()}")

In [ ]:
query="How do I clean and prepare data?"

results=collection.query(
    query_texts=[query],
    n_results=3
)

print("RESULT KEYS AVAILABLE:")
print(list(results.keys()))

In [ ]:
print(f"Query :'{query}'")
print("="*60)
print()

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distance = results['distances'][0]
matched_metadata = results['metadatas'][0]

for rank,(doc , doc_id,dist,meta)in enumerate(zip(matched_docs,matched_ids,matched_distance,matched_metadata)):
  print(f"Rank:{rank}|ID:{doc_id}|Distance:{dist:.2f}")
  print(f"Subject:{meta['subject']}|Topic:{meta['topic']}")
  print(f"Document:{doc}\n")

In [ ]:
filtered_results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"topic": "ETL"}
)



print(f"FILTERED QUERY: {query}")
print("Filter only Data Engineering documents")
print("="*60)

for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]),start=1):
  print(f"Rank: {rank} | Distance: {dist:.2f} | Subject: {meta['subject']}")
  print(f"{doc}\n")

In [ ]:
print("DISTANCE TO SIMILARITY CONVERSION")
print(f"{'Distance':<15}{'Similarity':<15}{'Interpretation':<20}")
distances=[0.05,0.20,0.40,0.65,0.90]
interpretations=["Near identical","Very similar","Related","Somewhat related","Not related"]
for dist,interp in zip(distances,interpretations):
  similarity=1-dist
  print(f"{dist:<15.2f}{similarity:<15.2f}{interp:<20}")

In [ ]:
pip install nbstripout